In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output


In [2]:
cap = cv2.VideoCapture(0)

# 读取第一帧作为背景
ret, bg_frame = cap.read()
if not ret:
    print("读取背景帧失败")
    cap.release()
else:
    bg_gray = cv2.cvtColor(bg_frame, cv2.COLOR_BGR2GRAY)
    
    try:
        for i in range(100):  # 跑 100 帧，想提前停止按 Jupyter 上面的 ⏹ 方块按钮
            ret, frame = cap.read()
            if not ret:
                break
            
            # 1. 灰度化
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            
            # 2. 帧差
            diff = cv2.absdiff(bg_gray, gray)
            
            # 3. 二值化
            _, binary = cv2.threshold(diff, 30, 255, cv2.THRESH_BINARY)
            
            # 4. 形态学去噪
            kernel = np.ones((5, 5), np.uint8)
            binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
            binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
            
            # 5. 找轮廓
            contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            # 6. 画框
            target_count = 0
            for cnt in contours:
                if cv2.contourArea(cnt) > 500:
                    x, y, w, h = cv2.boundingRect(cnt)
                    cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
                    target_count += 1
            
            # 7. 显示目标数量
            cv2.putText(frame, f"Targets: {target_count}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            
            # 8. 更新背景
            bg_gray = gray.copy()
            
            # 9. 在 Jupyter 里显示
            clear_output(wait=True)
            plt.figure(figsize=(10, 8))
            plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            plt.axis('off')
            plt.show()
            
    except KeyboardInterrupt:
        print("手动停止")
    
    cap.release()
    print("摄像头已释放")


手动停止
摄像头已释放


In [3]:
from pynq import Overlay
from pynq.lib.arduino import Arduino_DevMode
import time

# 加载 base overlay
ol = Overlay("base.bit")
ol.download()

# 配置 D4 为输出
arduino = Arduino_DevMode(ol.ARDUINO, "grove_buzzer", [4])
arduino.start()

print("蜂鸣器初始化完成")


AttributeError: Could not find IP or hierarchy ARDUINO in overlay

In [ ]:
# 蜂鸣器响 0.5 秒
arduino.write(4, 1)
time.sleep(0.5)
arduino.write(4, 0)


In [ ]:
from pynq import Overlay
from pynq.lib.arduino import Arduino_DevMode
import time

# 加载 base overlay
ol = Overlay("base.bit")
ol.download()

# 配置 D4 为输出
arduino = Arduino_DevMode(ol.ARDUINO, "grove_buzzer", [4])
arduino.start()

print("蜂鸣器初始化完成")